# Day05 下午个人项目：电商用户多维分析

**姓名：** 袁艳  
**专题方向：**  C 

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "袁艳"
TOPIC = "C"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 袁艳
专题： C
输入数据： c:\Users\34456\AppData\Roaming\JetBrains\PyCharm2026.1\scratches\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： c:\Users\34456\AppData\Roaming\JetBrains\PyCharm2026.1\scratches\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day05_analysis


In [2]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 袁艳
专题： C


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [3]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-6个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-6个月,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [4]:
# TODO 1：定义需要验收的核心字段
core_cols = [
   "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": df["CustomerID"].duplicated().sum(),
    "核心字段缺失数": df[core_cols].isnull().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}


# TODO 3：展示验收结果
display(validation)

{'行数': 5630,
 '列数': 22,
 'CustomerID重复数': np.int64(0),
 '核心字段缺失数': np.int64(0),
 'Churn取值': [0, 1]}

In [5]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行数据代表一名用户。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID是用户的唯一标识符（ID类变量），是分类变量而非连续数值变量，对其进行求平均没有业务意义，也得不到任何有价值的业务信息。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [6]:
# TODO：计算公共基础指标

overall_metrics = pd.DataFrame({
    "指标": [
        "用户数",
        "流失人数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用次数",
        "平均返现",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数"
    ],
    "数值": [
        len(df),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean()
    ]
})



# TODO：展示结果
display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [7]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"
print(f"总体流失率：{overall_churn_rate:.2%}")
print("检查点2通过")

总体流失率：16.84%
检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：前样本共有5630名用户，总体流失率为16.84%”，平均每人订单数为2.96单，平均距上次下单天数为4.46天。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [8]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "PreferedOrderCat"

# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby(segment_field)
    .agg(
        用户数=("CustomerID", "count"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        订单数中位数=("OrderCount", "median"),
        平均优惠券使用=("CouponUsed", "mean"),
        平均返现金额=("CashbackAmount", "mean")
    )
    .reset_index()
    .sort_values("用户数", ascending=False)
)

# TODO 3：重置索引、按业务意义排序并展示
display(segment_analysis)

当前专题： C
可选主分组字段： {'PreferedOrderCat'}


,PreferedOrderCat,用户数,流失人数,流失率,平均订单数,订单数中位数,平均优惠券使用,平均返现金额
3,Mobile Phone,2080,570,0.27,2.18,2.00,1.37,140.20
2,Laptop & Accessory,2050,210,0.10,2.77,2.00,1.65,167.22
0,Fashion,826,128,0.15,3.87,2.00,2.32,210.41
1,Grocery,410,20,0.05,4.60,2.00,2.19,266.23
4,Others,264,20,0.08,5.25,3.00,2.33,304.56


In [9]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同偏好品类的用户在用户规模、订单行为和流失风险上有何差异？

**数据现象：**

> TODO：Mobile Phone品类用户数最多（2080人，占37%），但流失率也最高（27%），平均订单数仅2.18单；而Grocery品类用户数最少（410人，占7%），但流失率最低（5%），平均订单数4.60单，表现最优。。

**可能解释：**

> TODO：Mobile Phone品类竞争激烈、手机的更新换代非常快等，用户品牌切换成本低、价格敏感度高，可能导致高流失率；Grocery品类属于日常刚需消费品，复购频率高、用户粘性强，因此流失率低。以上仅为基于现有数据观察到的相关模式，需要更多时间序列数据验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [10]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "PreferedOrderCat"
dim_2 = "CityTier"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = (
    df.groupby([dim_1, dim_2])
    .agg(
        用户数=("CustomerID", "count"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean")
    )
    .reset_index()
)


# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(
    lambda x: "小样本" if x < 30 else "可观察"
)



# TODO 4：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)
display(cross_analysis)

,PreferedOrderCat,CityTier,用户数,流失人数,流失率,平均订单数,样本提示
11,Mobile Phone,3,298,126,0.42,1.96,可观察
1,Fashion,2,24,8,0.33,2.33,小样本
10,Mobile Phone,2,130,40,0.31,2.05,可观察
9,Mobile Phone,1,1652,404,0.24,2.23,可观察
2,Fashion,3,316,68,0.22,4.36,可观察
8,Laptop & Accessory,3,928,150,0.16,2.77,可观察
14,Others,3,50,8,0.16,4.98,可观察
5,Grocery,3,130,16,0.12,4.70,可观察
0,Fashion,1,486,52,0.11,3.62,可观察
12,Others,1,188,12,0.06,5.26,可观察


In [11]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：Mobile Phone品类 + 三线城市，流失率高达42%，是流失风险最高的组合。

**该组合的用户数、流失率和比较对象：**

> TODO：用户数298人，流失率42%，明显高于该品类整体流失率（27%）和一线城市同品类流失率（24%）。

**是否存在小样本风险：**

> TODO：不存在。该组合样本量为298人，属于"可观察"范围，统计结果具有一定参考价值。但Fashion + CityTier 2（24人，小样本）和Others + CityTier 2（26人，小样本）需谨慎解读。

**为什么不能直接写成因果结论：**

> TODO：数据为截面数据，Mobile Phone在三线城市的高流失率可能受到多种混杂因素影响，如物流覆盖不完善、售后服务不便、当地竞争格局不同、用户价格敏感度更高等。当前数据无法控制这些变量，不能得出"三线城市导致手机品类用户流失"的因果结论。

## 任务5：输出统计报表（必做）

In [12]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [13]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 8)
通过：cross_analysis.csv，形状为(15, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在_____用户中，_____指标为____，与_____相比，对应证据表：_____

> TODO：在Mobile Phone品类用户（2080人）中，流失率指标为27%，与Grocery品类用户（410人，流失率5%）相比高出22个百分点，是流失风险最高的品类。对应证据表：segment_analysis。

### 结论2

> TODO：品类与城市等级交叉分析显示，Mobile Phone品类在三线城市的流失率高达42%（用户数298人），而同一品类在一线城市流失率仅为24%（用户数1652人），城市等级差异明显。对应证据表：cross_analysis。

### 结论3

> TODO：Grocery品类在一线城市表现最优，流失率仅2%（用户数266人），且平均订单数达4.59单，说明高频刚需品类结合良好城市基础设施能有效降低流失风险。对应证据表：cross_analysis。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：1.缺少订单金额和下单时间戳，因此无法计算GMV、客单价、复购间隔趋势及季节性波动等关键指标；2.数据为截面数据（单时间点），无法捕捉用户行为随时间的变化，所有"流失"判定基于静态标签；3.部分交叉组合样本量过小（如Fashion+CityTier 2仅24人），统计推断效力不足；4.返现（CashbackAmount）是返现金额，不是消费金额，不能用于衡量用户的实际购买力。


### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：建议：针对Mobile Phone品类在三线城市的高流失用户，可尝试：                           1.与三线城市本地手机维修/售后网点合作，提供专属服务保障;      2.推出"以旧换新"补贴活动，降低用户换机成本;3.优化三线城市手机品类的物流配送时效。验证方式：设计A/B测试实验：选取三线城市Mobile Phone品类近30天有活跃但可能流失的用户（建议300-500人），随机分为：A实验组：推送"以旧换新补贴+免费售后检测"专属权益；B对照组：无特殊干预。跟踪其后60天的复购率、订单数和流失率变化。同时收集优惠券核销率、客单价、售后满意度等数据全面评估效果。小范围试点验证有效后再全量推广。


## 拓展任务（选做）

In [14]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [15]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

1.Mobile Phone品类在三线城市流失率高达42%，而该品类整体流失率为27%，一线城市同品类流失率仅24%。同时，Grocery（生鲜杂货）品类在一线城市流失率仅2%，表现最优。这提示品类与城市等级的交互效应可能比单一维度更能解释流失风险。
2.检查点1（数据验收） 最有用。它强制检查了：数据形状是否为 (5630, 22)；CustomerID是否唯一；Churn是否只含0和1。这些基础检查帮助我提前发现数据质量问题，因为后面的数据都是在这个基础上的，避免后续分析出现系统性错误。
3."Mobile Phone品类在三线城市流失率高" 这条结论最容易被误解为"三线城市导致手机品类用户流失"。因为实际上这只是相关性，真正的因果链条可能更复杂：三线城市物流覆盖不足→售后不便→用户流失或者三线城市竞争格局不同→价格战激烈→用户比价流失或者是因为数据中未测量的第三变量同时影响了"三线城市"和"流失"。正确的说法应该是："数据分析发现，Mobile Phone品类在三线城市的流失率显著高于其他城市等级，值得进一步研究原因。"
4.订单金额（OrderAmount）
5.交叉分析表（cross_analysis）
原因：
对比效果直观：可以清晰看到Mobile Phone在三线城市流失率"鹤立鸡群"，一眼锁定高风险组合
验证小样本警告：用小样本组合用不同颜色或标记提示，避免过度解读
支持双维度解读：可以同时观察到品类差异（横向比较）和城市差异（纵向比较）
冲击力强：42%的柱状图比表格里的数字更有说服力，便于向业务方汇报